# Lab 18: Observability

Once the application is running in production, then we usually have no visibility into its performance. What are user's entering into the system? Is it giving right output? Are there areas with many failures that we need to work on?

To get answers to these kind of questions, we use *Observability tools*.

Some examples of Observability tools are:

- Langsmith
- Arize Phoenix
- Comet Opik
- BrainTrust
- and many more

In this lab, we will see how to integrate with Braintrust.

In order to run this lab, you will first need to create a free Braintrust account at https://www.braintrust.dev

## Sign up for Braintrust

- Sign up for Braintrust free plan - https://www.braintrust.dev/pricing
- You will be asked to create an organisation. Create one in your name
- After sign up it will ask if you want to add tracing for an existing application. Select that option and it will show you your API key. Copy it and keep it safe
- Rename the default project to "AI Evals"

Then open the `.env` file and add this line

`BRAINTRUST_API_KEY=sk-...`

Save the file and then continue with the notebook

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="gpt-5.1-chat-latest")

We will use the 3 step prompt chaining architecture for this example. Below are the prompts

In [ ]:
ANALYSIS_SYSTEM_PROMPT = """
You are an expert HR job description analyst specializing in inclusive hiring practices.

Analyze the provided job description for potential issues across these dimensions:

1. CLARITY: Identify sections with vague responsibilities, unclear expectations, or ambiguous requirements.
   Flag phrases like "various duties," "other tasks as assigned," or undefined acronyms.

2. JARGON: Flag unnecessarily technical language inappropriate for the role level.
   Consider whether terms would be understood by qualified candidates unfamiliar with internal terminology.

3. BIAS: Identify language that may discourage diverse candidates:
   - Gender-coded words (e.g., "rockstar," "ninja," "aggressive," "nurturing")
   - Age bias (e.g., "digital native," "recent graduate")
   - Exclusionary phrases (e.g., "culture fit," "work hard/play hard")
   - Excessive requirements (unnecessarily requiring degrees or years of experience)

4. MISSING INFORMATION: Note absent critical details:
   - Salary range or compensation structure
   - Work location/arrangement (remote/hybrid/onsite)
   - Reporting structure or team context
   - Clear distinction between required vs. preferred qualifications
   - Application process and timeline
   - Growth/development opportunities

5. SUMMARY: Provide 2-3 sentences describing overall quality and primary concerns.

For each issue you identify:
- Quote the exact problematic text
- Explain why it is problematic
- Suggest an improvement (when applicable)

Your output MUST be valid JSON that conforms exactly to the provided schema.
Do not include any text outside the JSON.

If information is missing, return empty arrays.
"""

ANALYSIS_USER_PROMPT = """
Analyze the following job description:

<job_description>
{job_description}
</job_description>

Return only JSON.
"""

REWRITE_SYSTEM_PROMPT = """
You are an expert HR editor specializing in rewriting job descriptions for clarity, inclusivity,
and accessibility.

You will receive:
1. The original job description.
2. A structured analysis of issues found in Step 1.

Your task is to rewrite ONLY the problematic sections, not the entire job description.

For each identified issue:
- Include the original problematic text (quoted exactly)
- Include the category (clarity, jargon, bias, or missing_information)
- Provide an improved, inclusive alternative that preserves meaning
- Maintain neutral, professional tone
- Ensure suggestions follow inclusive hiring practices

Return ONLY valid JSON matching the provided schema. Do not write any prose outside JSON.
"""

REWRITE_USER_PROMPT = """
Original Job Description:
-------------------------
{job_description}

Analysis Findings:
------------------
{analysis_json}

Rewrite ONLY the problematic sections using the schema.
Return only JSON.
"""

FINALISE_SYSTEM_PROMPT = """
You are an expert HR writer specializing in creating clear, concise, and inclusive job descriptions.

Your job is to produce the final polished version of the job description.

You will receive:
1. The original job description.
2. A list of rewritten sections (from Step 2).

Your tasks:
- Incorporate all improved rewritten sections into the original job description.
- Remove or replace the problematic text that was flagged in earlier steps.
- Maintain the original intent, structure, and role scope.
- Ensure clarity, inclusivity, and accessibility.
- Make tone consistent: professional, warm, and concise.
- Improve flow and readability where necessary.
- Do NOT invent new responsibilities, requirements, or benefits.

Return ONLY the final polished job description as plain text. Do not include JSON.
"""

FINALISE_USER_PROMPT = """
Original Job Description:
-------------------------
{job_description}

Rewritten Sections:
-------------------
{rewritten_sections_json}

Create the final polished job description by integrating the improvements.
Return only the final text.
"""

Now we create a function `revise_jd` that implements all the three steps in the chain and returns the final output

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel
from typing import Literal

class JDAnalysis(BaseModel):
    unclear_sections: list[str]
    jargon_terms: list[str]
    biased_language: list[str]
    missing_information: list[str]
    overall_summary: str

class RewrittenSection(BaseModel):
    category: Literal["clarity", "jargon", "bias", "missing_information"]
    original_text: str
    issue_explanation: str
    improved_text: str

class JDRewriteOutput(BaseModel):
    rewritten_sections: list[RewrittenSection]

def revise_jd(job_description):
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", ANALYSIS_SYSTEM_PROMPT),
        ("human", ANALYSIS_USER_PROMPT),
    ])
    chain = analysis_prompt | llm.with_structured_output(JDAnalysis)
    analysis = chain.invoke({"job_description": job_description})
    rewrite_prompt = ChatPromptTemplate.from_messages([
        ("system", REWRITE_SYSTEM_PROMPT),
        ("human", REWRITE_USER_PROMPT),
    ])
    rewrite_chain = rewrite_prompt | llm.with_structured_output(JDRewriteOutput)
    rewrite = rewrite_chain.invoke({"job_description": job_description, "analysis_json": analysis.model_dump_json()})
    finalise_prompt = ChatPromptTemplate.from_messages([
        ("system", FINALISE_SYSTEM_PROMPT),
        ("human", FINALISE_USER_PROMPT),
    ])
    finalise_chain = finalise_prompt | llm
    final_output = finalise_chain.invoke({
        "job_description": job_description, 
        "rewritten_sections_json": rewrite.model_dump_json()})
    return final_output.content

Let us execute the `revise_jd` function with a sample input and print the answer

In [ ]:
job_description = "We’re seeking a Forward Deployed Engineer. We want someone with 3+ years of software engineering experience with production systems. They should be rockstar programmers and problem solvers. They should have experience in a customer-facing technical role with a background in systems integration or professional services"
revised_job_description = revise_jd(job_description)
print(revised_job_description)

## Observability

Suppose the output above is not satisfactory. How do we know which of the three steps went wrong? We can only see the final output from the function and can't see the individual steps of each of the prompts.

This can be solved using *Tracing* which is a feature provided by observability tools.

To enable tracing, we import the Braintrust logger and integrate it into langchain.

In [ ]:
from braintrust import init_logger, traced
from braintrust.integrations.langchain import BraintrustCallbackHandler, set_global_handler

init_logger(project="AI Evals")
set_global_handler(BraintrustCallbackHandler())

Then we put the `@traced` decorator along with the name of the trace

In [ ]:
@traced(name="Review Job Description")
def revise_jd(job_description):
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", ANALYSIS_SYSTEM_PROMPT),
        ("human", ANALYSIS_USER_PROMPT),
    ])
    chain = analysis_prompt | llm.with_structured_output(JDAnalysis)
    analysis = chain.invoke({"job_description": job_description})
    rewrite_prompt = ChatPromptTemplate.from_messages([
        ("system", REWRITE_SYSTEM_PROMPT),
        ("human", REWRITE_USER_PROMPT),
    ])
    rewrite_chain = rewrite_prompt | llm.with_structured_output(JDRewriteOutput)
    rewrite = rewrite_chain.invoke({"job_description": job_description, "analysis_json": analysis.model_dump_json()})
    finalise_prompt = ChatPromptTemplate.from_messages([
        ("system", FINALISE_SYSTEM_PROMPT),
        ("human", FINALISE_USER_PROMPT),
    ])
    finalise_chain = finalise_prompt | llm
    final_output = finalise_chain.invoke({
        "job_description": job_description, 
        "rewritten_sections_json": rewrite.model_dump_json()})
    return final_output.content

Let us run the function again, this time with observability tracing enabled.

In [ ]:
job_description = "We’re seeking a Forward Deployed Engineer. We want someone with 3+ years of software engineering experience with production systems. They should be rockstar programmers and problem solvers. They should have experience in a customer-facing technical role with a background in systems integration or professional services"
revised_job_description = revise_jd(job_description)
print(revised_job_description)

## Viewing Trace output

Now we can go to https://www.braintrust.dev and navigate to the "Logs" section to see our trace there. Click the trace and we can see all the individual spans that make up the trace